In [ ]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver

In [ ]:
# 모듈 불러오기
%run /content/AIFFEL_final_project_peekabook/research/src/state/state_v1.ipynb
%run /content/AIFFEL_final_project_peekabook/research/src/db/qdrant.py
%run /content/AIFFEL_final_project_peekabook/research/src/embedding/embedder.py

In [ ]:
import os
from google.colab import userdata

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Set the HCX API key environment variable
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')

# Set the Qdrant API key environment variable
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

# ── 초기화 ──────────────────────────────
embedder = LocalEmbedder("BAAI/bge-m3")   # 임베딩
db = QdrantDB(vector_size=1024)           # Qdrant 연결

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")     # OpenAI LLM
# llm = ChatClovaX(model="HCX-005", temperature=0)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
import json


def simple_rag_node(state: CRSState) -> dict:
    # Qdrant에서 후보 책 5개 검색 -> retrieved_books 저장
    summary = state.get("summary", "")
    query_vector = embedder.embed(summary)

    results = db.search("books_v1", query_vector, limit=5, threshold=0.5)

    # 다음 노드로 넘길 형태로 정리
    retrieved_books = [
        {
            "isbn": r.payload.get("isbn"),
            "title": r.payload.get("title"),
            "author": r.payload.get("author"),
            "book_intro": r.payload.get("book_intro"),
        }
        for r in results
    ]

    return {"retrieved_books": retrieved_books}


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"소개: {b['book_intro']}"
        for b in books
    ])

    # 프롬프트 직접 정의
    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "reason": "추천 이유 2~3문장"}}
]
"""
    )

    chain = rag_prompt | llm
    response = chain.invoke({
        "context": context,
        "summary": summary
    })

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        # 파싱 실패해도 LLM 응답 그대로 반환
        recommendations = response.content

    # recommendations 출력 확인
    # print(recommendations)

    return {
        "messages": [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

In [ ]:
# # ── TEST 실행 ────────────────────────────────
# graph = StateGraph(BookAgentState)
# graph.add_node("rag", simple_rag_node)
# graph.add_node("llm", rag_llm_node)
# graph.add_edge(START, "rag")
# graph.add_edge("rag", "llm")
# graph.add_edge("llm", END)
# app = graph.compile()

# result = app.invoke({
#     "messages": [HumanMessage(content="번아웃이 심한데 위로가 되는 소설 추천해줘")],
#     "retrieved_books": [],
#     "recommendations": []
# })

# print(result["messages"][-1].content)

[
    {
        "title": "내가 미친 8주간의 기록",
        "author": "에바 로만",
        "isbn": "9788965702047",
        "reason": "우울증, 무기력, 번아웃 증후군 등을 다루며, 주인공의 이야기를 통해 독자에게 공감과 위로를 제공합니다. 또한 현대인들이 겪는 심리적 고통을 감각적으로 표현하여 쉽게 읽을 수 있는 소설입니다."
    },
    {
        "title": "상처받은 내면아이 치유",
        "author": "존 브래드쇼",
        "isbn": "9788999731419",
        "reason": "어린 시절의 상처를 치유하는 방법을 제시하며, 번아웃으로 인한 심리적 어려움을 겪는 사람들에게 위로와 도움을 줍니다. 실제 사례를 바탕으로 쓰여졌으며, 자신의 내면을 탐색하고 치유하는 과정에 초점을 맞춘 책입니다."
    },
    {
        "title": "재가 된 여자들",
        "author": ["에밀리 나고스키", "어밀리아 나고스키 피터슨"],
        "isbn": "9788986022759",
        "reason": "여성들이 겪는 번아웃을 새로운 시각으로 바라보며, 사회적 압박과 역할 강요로 인한 감정적 소진을 다룹니다. 이를 극복하기 위한 구체적인 방법들을 제시하여 독자에게 실질적인 도움을 줄 수 있는 책입니다."
    }
]


In [ ]:
# result["recommendations"]